# Question 2 — Augmenter le prix de 10 % changerait-il les ventes ?

## Objectif
Évaluer, à partir des événements nettoyés, si une hausse de 10 % du prix moyen est associée à une évolution des ventes.

**Méthodes de data mining utilisées :**
- Régression linéaire
- KNN (régression)

Les événements dont `billets_vendus` est inconnu sont exclus, conformément au preprocessing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Données nettoyées
df = pd.read_csv(
    '../data/processed/evenements_clean.csv',
    parse_dates=['date']
)

# Q1-Q4 : les 3 événements avec ventes inconnues sont exclus
df = df.dropna(subset=['billets_vendus', 'taux_remplissage']).copy()

# Capacité = configuration de salle, donc variable catégorielle
df['capacite_cat'] = df['capacite'].astype(str)

print("Nombre d'événements analysés :", len(df))
display(df.head())


## 1. Comprendre la relation prix → ventes

Avant de modéliser, on regarde visuellement si les événements plus chers ont tendance à vendre davantage ou moins de billets.

⚠️ Attention : les événements complets sont plafonnés par la capacité. Le nombre de billets vendus mesure donc parfois la capacité de la salle plutôt que la demande réelle.


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df['prix_moyen_eur'], df['billets_vendus'], alpha=0.7)
plt.xlabel("Prix moyen (€)")
plt.ylabel("Billets vendus")
plt.title("Prix moyen et nombre de billets vendus")
plt.grid(alpha=0.2)
plt.show()


## 2. Régression linéaire

La régression linéaire permet d'estimer les ventes en tenant compte simultanément du prix, de la notoriété, du genre, du jour et de la capacité.

La capacité est encodée comme catégorie car le projet identifie trois configurations de salle : 6 500, 9 800 et 15 800 places.


In [ ]:
features = [
    'prix_moyen_eur',
    'notoriete_artiste',
    'genre',
    'jour_semaine',
    'capacite_cat'
]
target = 'billets_vendus'

X = df[features]
y = df[target]

categorical = ['genre', 'jour_semaine', 'capacite_cat']
numeric = ['prix_moyen_eur', 'notoriete_artiste']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
    ]
)

modele_lineaire = Pipeline([
    ('preparation', preprocessor),
    ('modele', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

modele_lineaire.fit(X_train, y_train)
pred_test = modele_lineaire.predict(X_test)

print("Régression linéaire")
print("MAE :", round(mean_absolute_error(y_test, pred_test), 2))
print("R²  :", round(r2_score(y_test, pred_test), 3))


## 3. KNN

KNN estime les ventes d'un événement à partir d'événements similaires dans les données.

On standardise les variables numériques et on encode les variables catégorielles avant de calculer les distances.


In [ ]:
modele_knn = Pipeline([
    ('preparation', preprocessor),
    ('modele', KNeighborsRegressor(n_neighbors=5))
])

modele_knn.fit(X_train, y_train)
pred_knn = modele_knn.predict(X_test)

print("KNN (k=5)")
print("MAE :", round(mean_absolute_error(y_test, pred_knn), 2))
print("R²  :", round(r2_score(y_test, pred_knn), 3))


In [ ]:
comparaison = pd.DataFrame({
    'Méthode': ['Régression linéaire', 'KNN (k=5)'],
    'MAE': [
        mean_absolute_error(y_test, pred_test),
        mean_absolute_error(y_test, pred_knn)
    ],
    'R²': [
        r2_score(y_test, pred_test),
        r2_score(y_test, pred_knn)
    ]
})

display(comparaison.round(3))


## 4. Simulation d'une hausse de prix de 10 %

On garde les autres caractéristiques de chaque événement identiques et on augmente uniquement `prix_moyen_eur` de 10 %.

La comparaison est faite avec les deux modèles.


In [ ]:
df_sim = df.copy()
df_sim['prix_moyen_eur'] = df_sim['prix_moyen_eur'] * 1.10

X_actuel = df[features]
X_plus10 = df_sim[features]

pred_actuel_lr = modele_lineaire.predict(X_actuel)
pred_plus10_lr = modele_lineaire.predict(X_plus10)

pred_actuel_knn = modele_knn.predict(X_actuel)
pred_plus10_knn = modele_knn.predict(X_plus10)

resultats = df[['evenement_id', 'prix_moyen_eur', 'billets_vendus']].copy()

resultats['ventes_pred_lr_avant'] = pred_actuel_lr
resultats['ventes_pred_lr_plus10'] = pred_plus10_lr
resultats['variation_lr'] = (
    resultats['ventes_pred_lr_plus10']
    - resultats['ventes_pred_lr_avant']
)

resultats['variation_pct_lr'] = (
    resultats['variation_lr']
    / resultats['ventes_pred_lr_avant']
    * 100
)

resultats['ventes_pred_knn_avant'] = pred_actuel_knn
resultats['ventes_pred_knn_plus10'] = pred_plus10_knn
resultats['variation_knn'] = (
    resultats['ventes_pred_knn_plus10']
    - resultats['ventes_pred_knn_avant']
)

resultats['variation_pct_knn'] = (
    resultats['variation_knn']
    / resultats['ventes_pred_knn_avant']
    * 100
)

display(resultats.round(2))


## 5. Résultat global de la simulation

On compare les ventes moyennes prédites avant et après la hausse de 10 %.


In [ ]:
bilan = pd.DataFrame({
    'Modèle': ['Régression linéaire', 'KNN'],
    'Ventes moyennes avant': [
        pred_actuel_lr.mean(),
        pred_actuel_knn.mean()
    ],
    'Ventes moyennes après +10%': [
        pred_plus10_lr.mean(),
        pred_plus10_knn.mean()
    ]
})

bilan['Variation moyenne (%)'] = (
    (bilan['Ventes moyennes après +10%']
     - bilan['Ventes moyennes avant'])
    / bilan['Ventes moyennes avant']
    * 100
)

display(bilan.round(2))


## 6. Effet sur le chiffre d'affaires

Le chiffre d'affaires prédit est approximé par :

**prix × nombre de billets vendus**

On compare donc le CA avant et après la hausse.


In [ ]:
ca_lr_avant = (
    df['prix_moyen_eur'] * pred_actuel_lr
).mean()

ca_lr_apres = (
    df_sim['prix_moyen_eur'] * pred_plus10_lr
).mean()

ca_knn_avant = (
    df['prix_moyen_eur'] * pred_actuel_knn
).mean()

ca_knn_apres = (
    df_sim['prix_moyen_eur'] * pred_plus10_knn
).mean()

ca = pd.DataFrame({
    'Modèle': ['Régression linéaire', 'KNN'],
    'CA moyen avant (€)': [ca_lr_avant, ca_knn_avant],
    'CA moyen après +10% (€)': [ca_lr_apres, ca_knn_apres]
})

ca['Variation CA (%)'] = (
    (ca['CA moyen après +10%']
     - ca['CA moyen avant'])
    / ca['CA moyen avant']
    * 100
)

display(ca.round(2))


## 7. Conclusion

La décision doit être basée sur les résultats des deux méthodes et sur la comparaison de leurs performances.

**Point important :** cette analyse est une simulation basée sur des données observationnelles. Elle mesure une association entre le prix et les ventes, mais ne permet pas à elle seule de prouver qu'une hausse de prix provoquerait causalement la variation observée.

De plus, les événements complets sont soumis à un effet plafond : leurs ventes ne peuvent pas dépasser la capacité. Une hausse de prix peut donc avoir un effet sur la demande réelle qui n'apparaît pas dans les ventes observées.


### Recommandation

Si les deux modèles donnent une baisse des ventes après +10 %, la hausse de prix doit être considérée avec prudence.

Si le chiffre d'affaires augmente malgré cette baisse, le +10 % peut rester intéressant économiquement, mais il faut confirmer le résultat avec une expérimentation ou des données historiques comparables avant de généraliser.
